In [ ]:
%pip insatall -q langchain langchain-huggingface langchain-community langchain-core langchain-text-splitters docx2txt bitsandbytes langchain-chroma

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 상용화된 임베딩 모델보다 오픈소스가 효율이 좀 떨어지기 때문에 청크 사이즈를 줄여서
# 좀 더 의미 단위로 잘 잘려 임베딩 성능이 더 나아진다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

loader = Docx2txtLoader("./tax_docs/tax_with_markdown.docx")
document_list = loader.load_and_split(text_splitter)

In [ ]:
# 허깅 페이스 사용 시 허깅페이스 액세스 토큰 가져와야함
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# HuggingFace의 임베딩 모델을 사용
embedding = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large-instruct")

In [ ]:
from langchain_chroma import Chroma

collection_name = "chroma_tax"

database = Chroma.from_documents(document_list, embedding, collection_name=collection_name, persist_directory="./chroma_huggingface")

In [ ]:
# 양자화
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# 허깅페이스 LLM모델 불러오기
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

chat_model =  HuggingFacePipeline.from_model_id(
    # LG에서 출시한 모델
    model_id="LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=1024,
        do_sample=False,
        repetition_penalty=1.03,
    ),
    model_kwargs={"quantization_config": quantization_config},
)


In [ ]:
llm = ChatHuggingFace(llm=chat_model)

In [ ]:
# create retrieval chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic import hub

retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")
retriever = database.as_retriever(search_kwargs={"k": 3})


In [ ]:
query = "연봉 5천만원인 거주자의 소득세는?"

retrieved_docs = retriever.invoke(query)

In [ ]:
combine_docs_chain = create_stuff_documents_chain(
    llm, retrieval_qa_chat_prompt
)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
ai_message = retrieval_chain.invoke({"input": query})

In [ ]:
# llm 답변 테스트
test_ai_message = llm.invoke("LG는 어떤 회사인가요?")